In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("earthquake_engineered.csv")
feature_cols = ['Latitude', 'Longitude', 'Depth', 'Year', 'Month', 'DayOfWeek', 'RollingMag_30D']
X = df[feature_cols].values
y = df['Magnitude'].values

# shuffle=False keeps this a time-respecting split, consistent with the rolling feature above
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [2]:
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "Support Vector Regression": SVR(),
}

In [3]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    results[name] = {"MSE": mse, "RMSE": mse ** 0.5, "R2": r2_score(y_test, pred)}
    print(f"{name:28s} MSE={mse:.4f}  RMSE={mse**0.5:.4f}  R2={r2_score(y_test, pred):.4f}")

Random Forest Regressor      MSE=0.1889  RMSE=0.4347  R2=0.0045


Gradient Boosting Regressor  MSE=0.1772  RMSE=0.4209  R2=0.0664


Support Vector Regression    MSE=0.1999  RMSE=0.4471  R2=-0.0533


In [4]:
results_df = pd.DataFrame(results).T
results_df.to_csv("regression_results.csv")
print(results_df)

# Note: low R2 here is expected and worth stating as a limitation, not hidden -
# magnitude is governed by fault mechanics that lat/long/depth/time alone don't capture well.

                                  MSE      RMSE        R2
Random Forest Regressor      0.188933  0.434665  0.004519
Gradient Boosting Regressor  0.177194  0.420945  0.066371
Support Vector Regression    0.199910  0.447113 -0.053316
